In [1]:
import requests
import pandas as pd
from datetime import datetime

HEADERS = {"User-Agent": "chess-dashboard/1.0 student-project"}
USERNAME = "888yashu888"

In [2]:
def get_archives(username):
    url = f"https://api.chess.com/pub/player/{username}/games/archives"
    return requests.get(url, headers=HEADERS).json().get("archives", [])

def get_games_from_archive(url):
    return requests.get(url, headers=HEADERS).json().get("games", [])

In [3]:
archives = get_archives(USERNAME)

all_games = []
for url in archives:
    print(f"Fetching {url}...")
    all_games.extend(get_games_from_archive(url))

print(f"Total games fetched: {len(all_games)}")

Fetching https://api.chess.com/pub/player/888yashu888/games/2023/06...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/07...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/08...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/09...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/10...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/11...
Fetching https://api.chess.com/pub/player/888yashu888/games/2023/12...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/01...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/02...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/03...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/04...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/05...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/06...
Fetching https://api.chess.com/pub/player/888yashu888/games/2024/07...
Fetchi

In [4]:
rows = []
for g in all_games:
    if g.get("rules") != "chess":
        continue

    white = g["white"]["username"].lower()
    my_color = "white" if white == USERNAME.lower() else "black"
    opp_color = "black" if my_color == "white" else "white"

    my_result = g[my_color]["result"]
    if my_result == "win":
        outcome = "Win"
    elif my_result in ["checkmated", "resigned", "timeout", "abandoned"]:
        outcome = "Loss"
    else:
        outcome = "Draw"

    accuracies = g.get("accuracies", {})
    eco_url = g.get("eco", "")
    opening = eco_url.split("/")[-1].replace("-", " ") if eco_url else "Unknown"
    end_time = datetime.utcfromtimestamp(g["end_time"])

    rows.append({
        "date":       end_time.strftime("%Y-%m-%d"),
        "month":      end_time.strftime("%Y-%m"),
        "time_class": g.get("time_class", "unknown"),
        "color":      my_color,
        "outcome":    outcome,
        "my_rating":  g[my_color]["rating"],
        "opp_rating": g[opp_color]["rating"],
        "opp_name":   g[opp_color]["username"],
        "accuracy":   accuracies.get(my_color, None),
        "opening":    opening,
        "url":        g.get("url", ""),
    })

C:\Users\ysriv\AppData\Local\Temp\ipykernel_15176\4027212806.py:21: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  end_time = datetime.utcfromtimestamp(g["end_time"])


In [5]:
df = pd.DataFrame(rows)
df.to_csv("chess_games.csv", index=False)
print(f"Saved {len(df)} games to chess_games.csv")

Saved 3797 games to chess_games.csv
